# 🛡️ Robustness Analysis on Roman-empire

**Milestone 3 — Analysis 2: Robustness Analysis**

Test stability across `GATv2`, `GraphSAGE`, `H2GCN`, and `MLP` on **Roman-empire** (largest dataset):
1. **Edge Dropout**: Remove 10%–50% of undirected edges
2. **Feature Noise**: Add Gaussian noise (σ = 0.1–1.0)
3. **Label Scarcity**: Train with 5%–100% of training labels

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.datasets import load_dataset
from src.models import create_model
from src.evaluate import (
    evaluate_with_edge_dropout,
    evaluate_with_feature_noise,
    evaluate_with_label_scarcity,
)
from src.utils import set_seed, get_device

device = get_device()
set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150
os.makedirs('../results/plots', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

ROBUST_MODELS = ['gatv2', 'graphsage', 'h2gcn', 'mlp']
MODEL_DISPLAY = {'gatv2': 'GATv2', 'graphsage': 'GraphSAGE', 'h2gcn': 'H2GCN', 'mlp': 'MLP'}
MODEL_COLORS = {'gatv2': '#1b4965', 'graphsage': '#2a9d8f', 'h2gcn': '#e76f51', 'mlp': '#ca6702'}

## 1. Load Roman-empire & Trained Checkpoints

In [ ]:
data = load_dataset('roman-empire', root='../data')
print(f"Dataset: {data.dataset_name} ({data.num_nodes} nodes)")

models = {}
checkpoints = {}
for mname in ROBUST_MODELS:
    ckpt_path = f'../results/models/roman-empire/{mname}/seed_42.pt'
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    params = dict(checkpoint.get('model_kwargs', {}))
    model = create_model(mname, data.num_node_features, data.num_classes, **params).to(device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    models[mname] = model
    checkpoints[mname] = checkpoint
    print(f"Loaded {MODEL_DISPLAY[mname]} (val_acc: {checkpoint.get('best_val_accuracy', 0.0):.4f})")

## 2. Edge Dropout & Feature Noise

In [ ]:
edge_drop_rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
noise_stds = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
edge_results, noise_results = {}, {}

for mname in ROBUST_MODELS:
    e_means, e_stds = [], []
    for rate in edge_drop_rates:
        m, s = evaluate_with_edge_dropout(models[mname], data, rate, device, num_runs=5)
        e_means.append(m); e_stds.append(s)
    edge_results[mname] = {'means': e_means, 'stds': e_stds}

    n_means, n_stds = [], []
    for std in noise_stds:
        m, s = evaluate_with_feature_noise(models[mname], data, std, device, num_runs=5)
        n_means.append(m); n_stds.append(s)
    noise_results[mname] = {'means': n_means, 'stds': n_stds}
    print(f"{MODEL_DISPLAY[mname]}: clean={e_means[0]:.4f}, 50% edge drop={e_means[-1]:.4f}, noise=1.0 -> {n_means[-1]:.4f}")

## 3. Label Scarcity

In [ ]:
label_fractions = [0.05, 0.1, 0.2, 0.4, 1.0]
scarcity_results = {}

for mname in ROBUST_MODELS:
    ckpt = checkpoints[mname]
    params = dict(ckpt.get('model_kwargs', {}))
    lr = float(ckpt.get('lr', 0.005))
    wd = float(ckpt.get('weight_decay', 5e-4))
    accs, f1s = [], []
    for frac in label_fractions:
        acc, f1 = evaluate_with_label_scarcity(
            mname, data, device, train_fraction=frac, seed=42,
            max_epochs=300, patience=30, lr=lr, weight_decay=wd, **params
        )
        accs.append(acc); f1s.append(f1)
    scarcity_results[mname] = {'accs': accs, 'f1s': f1s}
    print(f"{MODEL_DISPLAY[mname]}: 5% labels={accs[0]:.4f}, 100% labels={accs[-1]:.4f}")

## 4. Combined 3-Panel Robustness Plot & Summary Table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)

for mname in ROBUST_MODELS:
    c = MODEL_COLORS[mname]; lbl = MODEL_DISPLAY[mname]
    axes[0].errorbar(edge_drop_rates, edge_results[mname]['means'], yerr=edge_results[mname]['stds'], marker='o', capsize=3, color=c, label=lbl)
    axes[1].errorbar(noise_stds, noise_results[mname]['means'], yerr=noise_results[mname]['stds'], marker='o', capsize=3, color=c, label=lbl)
    axes[2].plot([f * 100 for f in label_fractions], scarcity_results[mname]['accs'], marker='D', color=c, label=lbl)

axes[0].set_title('Edge Dropout', fontweight='bold'); axes[0].set_xlabel('Fraction of edges removed')
axes[1].set_title('Feature Noise', fontweight='bold'); axes[1].set_xlabel('Gaussian noise std (σ)')
axes[2].set_title('Label Scarcity', fontweight='bold'); axes[2].set_xlabel('Training Labels Used (%)')
for ax in axes:
    ax.set_ylabel('Test Accuracy'); ax.set_ylim(0, 1.0); ax.legend(fontsize=8); ax.grid(alpha=0.25)

fig.suptitle('Robustness Analysis on Roman-empire', fontweight='bold')
fig.savefig('../results/plots/roman_robustness.png', dpi=180)
plt.show()

summary_rows = []
for mname in ROBUST_MODELS:
    clean = edge_results[mname]['means'][0]
    summary_rows.append({
        'Model': MODEL_DISPLAY[mname],
        'Clean Acc': f"{clean:.4f}",
        '50% Edge Drop Acc': f"{edge_results[mname]['means'][-1]:.4f}",
        'Noise σ=1.0 Acc': f"{noise_results[mname]['means'][-1]:.4f}",
        '5% Labels Acc': f"{scarcity_results[mname]['accs'][0]:.4f}",
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../results/tables/robustness_summary.csv', index=False)
summary_df